<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 00. word2vec SGNS

## Distributed Representations of Words and Phrases and their Compositionality

- **원 논문:** [Distributed Representations of Words and Phrases and their Compositionality](https://arxiv.org/abs/1310.4546)
- **저자 / 발표:** Tomas Mikolov, Ilya Sutskever, Kai Chen, Greg Corrado, Jeffrey Dean · NeurIPS 2013 (2013)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** skip-gram context pair, negative sampling, unigram^(3/4) noise를 재현한다.

**축소하거나 생략한 부분:** 수십억 token과 300차원 embedding 대신 로컬 360문장과 16차원을 사용하고, 288/72개의 분리된 문장 행으로 학습/평가한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2 Fig. 1, Eq. (1): nearby-word prediction | Task 1 window pairs | pair 수 |
| §2.2 Eq. (4): NEG objective | Task 2–3 | positive/negative logits |
| §2.2: Pn(w)=U(w)^(3/4)/Z | Task 1 | distribution sum |
| §5: additive/compositional representations | Task 3 | embedding scores |

## 핵심 아이디어와 수식

$$\log\sigma(v_{w_O}^{\top}v_{w_I})+\sum_{i=1}^{k}\mathbb{E}_{w_i\sim P_n}\log\sigma(-v_{w_i}^{\top}v_{w_I})$$

**기호 해설:** w_I는 중심어, w_O는 실제 문맥어, w_i는 noise, k는 negative 개수다.

**코드 대응:** Task 1이 pair/noise 분포, Task 2가 두 embedding table, Task 3이 SGNS loss를 구현한다.

**입력과 출력 shape:** center [N], positive [N], negative [N,k] → dot logits [N], [N,k] → scalar loss.

**포트폴리오 구현 범위:** subsampling과 phrase detection은 제외하지만 Eq. (4)와 3/4 noise 분포는 보존한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2 Fig. 1, Eq. (1): nearby-word prediction
- **노트북에서 구현할 부분:** Task 1 window pairs
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** pair 수

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class SGNSConfig:
    embedding_dim: int = 16
    window_size: int = 1
    negatives: int = 3
    noise_exponent: float = 0.75
    learning_rate: float = 0.04
    steps: int = 18


def build_skipgram_pairs(token_lines, window_size):
    pairs = []
    for line in token_lines:
        for center_index, center in enumerate(line):
            left = max(0, center_index - window_size)
            right = min(len(line), center_index + window_size + 1)
            for context_index in range(left, right):
                if context_index != center_index:
                    pairs.append((center, line[context_index]))
    return pairs


def build_noise_distribution(token_lines, vocab_size, exponent):
    flattened = [token for line in token_lines for token in line]
    counts = torch.bincount(
        torch.tensor(flattened),
        minlength=vocab_size,
    ).float()
    powered = counts.pow(exponent)
    return powered / powered.sum()


config = SGNSConfig()
token_lines = [
    [stoi[word.lower()] for word in sentence.split()] for sentence in sentences
]
split_generator = torch.Generator().manual_seed(17)
sentence_order = torch.randperm(
    len(token_lines),
    generator=split_generator,
)
train_sentence_index = sentence_order[:288]
test_sentence_index = sentence_order[288:]
assert not bool(torch.isin(train_sentence_index, test_sentence_index).any())
train_token_lines = [token_lines[index] for index in train_sentence_index.tolist()]
test_token_lines = [token_lines[index] for index in test_sentence_index.tolist()]
train_pairs = build_skipgram_pairs(
    train_token_lines,
    config.window_size,
)
test_pairs = build_skipgram_pairs(
    test_token_lines,
    config.window_size,
)
noise_probability = build_noise_distribution(
    train_token_lines,
    VOCAB_SIZE,
    config.noise_exponent,
)
train_centers = torch.tensor(
    [pair[0] for pair in train_pairs],
    device=DEVICE,
)
train_contexts = torch.tensor(
    [pair[1] for pair in train_pairs],
    device=DEVICE,
)
test_centers = torch.tensor(
    [pair[0] for pair in test_pairs],
    device=DEVICE,
)
test_contexts = torch.tensor(
    [pair[1] for pair in test_pairs],
    device=DEVICE,
)
assert len(train_pairs) > 1000
assert len(test_pairs) > 100
assert torch.allclose(
    noise_probability.sum(),
    noise_probability.new_tensor(1.0),
)

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.2 Eq. (4): NEG objective
- **노트북에서 구현할 부분:** Task 2–3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** positive/negative logits

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
class SGNSModel(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.input_embedding = nn.Embedding(vocab_size, config.embedding_dim)
        self.output_embedding = nn.Embedding(vocab_size, config.embedding_dim)
        nn.init.normal_(self.input_embedding.weight, std=0.1)
        nn.init.zeros_(self.output_embedding.weight)

    def positive_score(self, centers, positives):
        center_vectors = self.input_embedding(centers)
        positive_vectors = self.output_embedding(positives)
        return (center_vectors * positive_vectors).sum(dim=-1)

    def negative_scores(self, centers, negatives):
        center_vectors = self.input_embedding(centers)
        negative_vectors = self.output_embedding(negatives)
        return torch.einsum("bd,bkd->bk", center_vectors, negative_vectors)

    def forward(self, centers, positives, negatives):
        positive = self.positive_score(centers, positives)
        negative = self.negative_scores(centers, negatives)
        return positive, negative


train_negative_generator = torch.Generator().manual_seed(7)
test_negative_generator = torch.Generator().manual_seed(8)
train_negatives = torch.multinomial(
    noise_probability,
    len(train_centers) * config.negatives,
    replacement=True,
    generator=train_negative_generator,
).view(-1, config.negatives)
test_negatives = torch.multinomial(
    noise_probability,
    len(test_centers) * config.negatives,
    replacement=True,
    generator=test_negative_generator,
).view(-1, config.negatives)
train_negatives = train_negatives.to(DEVICE)
test_negatives = test_negatives.to(DEVICE)
model = SGNSModel(VOCAB_SIZE, config).to(DEVICE)
positive_logits, negative_logits = model(
    train_centers[:8],
    train_contexts[:8],
    train_negatives[:8],
)
assert positive_logits.shape == (8,)
assert negative_logits.shape == (8, 3)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §5: additive/compositional representations
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** embedding scores

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def sgns_loss(positive_logits, negative_logits):
    positive_term = F.logsigmoid(positive_logits)
    negative_term = F.logsigmoid(-negative_logits).sum(dim=1)
    return -(positive_term + negative_term).mean()


def train_sgns_step(model, optimizer, centers, contexts, negatives):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    positive, negative = model(centers, contexts, negatives)
    loss = sgns_loss(positive, negative)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_sgns(model, centers, contexts, negatives):
    model.eval()
    with torch.no_grad():
        positive, negative = model(centers, contexts, negatives)
    return float(positive.mean()), float(negative.mean())


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(
        train_sgns_step(
            model,
            optimizer,
            train_centers,
            train_contexts,
            train_negatives,
        )
    )
heldout_positive_score, heldout_noise_score = evaluate_sgns(
    model,
    test_centers,
    test_contexts,
    test_negatives,
)
assert min(losses[3:]) < losses[0]
assert heldout_positive_score > heldout_noise_score
figure, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(losses)
axes[0].set_title("SGNS loss")
axes[1].bar(
    ["positive", "noise"],
    [heldout_positive_score, heldout_noise_score],
)
axes[1].set_title("held-out dot scores")
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 수십억 token과 300차원 embedding 대신 로컬 360문장과 16차원을 사용하고, 288/72개의 분리된 문장 행으로 학습/평가한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.